In [1]:
# Realtime sine synth with low-latency callback + ipywidgets UI
# Requirements:
#   pip install sounddevice numpy ipywidgets
#   In Jupyter: enable widgets extension if needed (e.g., jupyter nbextension enable --py widgetsnbextension)

import numpy as np
import sounddevice as sd
import threading
from ipywidgets import FloatSlider, ToggleButton, VBox, HBox, Label, Layout, Dropdown, HTML
from IPython.display import display


In [2]:
# Realtime N-parameter synth with class-based generators + normalized sliders [0,1]
# pip install sounddevice numpy ipywidgets

# --------- utils ---------
def exp_map01(x, lo, hi):
    """Exponential map from [0,1] -> [lo,hi] (lo>0)."""
    x = np.clip(x, 0.0, 1.0)
    return lo * (hi / lo) ** x

# --------- generator base ---------
class BaseGenerator:
    """
    Interface for realtime generators with N normalized params in [0,1].
    Subclasses should define:
      - self.param_labels: list[str]  (length = number of params)
      - set_params(norm_params: list[float])  -> map [0,1] to semantic values + store state
      - generate(frames:int, sr:int) -> np.ndarray float32, mono
      - formatted_readouts() -> list[str]   (one per param, human-readable)
    """
    param_labels = []  # override in subclass

    def __init__(self, init_norm_params=None):
        self.norm_params = list(init_norm_params) if init_norm_params else [0.5]*len(self.param_labels)

    def num_params(self):
        return len(self.param_labels)

    def set_params(self, norm_params):
        # Save normalized parameters (clip for safety); subclasses should map to semantic.
        self.norm_params = [float(np.clip(v, 0.0, 1.0)) for v in norm_params]

    def generate(self, frames, sr):
        raise NotImplementedError

    def formatted_readouts(self):
        # By default just show normalized values; subclasses usually override.
        return [f"{label}: {v:.3f}" for label, v in zip(self.param_labels, self.norm_params)]

In [3]:
# --------- example generators ---------
class SineGenerator(BaseGenerator):
    """
    Sine oscillator.
      p[0] -> frequency in [20, 2000] Hz (exp)
      p[1] -> amplitude in [0, 1] (linear)
    """
    param_labels = ["Freq (Hz)", "Amp"]

    def __init__(self, f_lo=20.0, f_hi=2000.0, init_norm_params=None):
        super().__init__(init_norm_params=init_norm_params or [0.5, 0.2])
        self.f_lo = max(1e-3, float(f_lo))
        self.f_hi = max(self.f_lo, float(f_hi))
        self.phase = 0.0
        self.twopi = 2.0 * np.pi
        # initialize semantic
        self.freq = exp_map01(self.norm_params[0], self.f_lo, self.f_hi)
        self.amp  = self.norm_params[1]

    def set_params(self, norm_params):
        super().set_params(norm_params)
        self.freq = exp_map01(self.norm_params[0], self.f_lo, self.f_hi)
        self.amp  = self.norm_params[1]

    def generate(self, frames, sr):
        if self.amp <= 0.0 or self.freq <= 0.0:
            return np.zeros(frames, dtype=np.float32)
        inc = self.twopi * self.freq / sr
        idx = self.phase + inc * np.arange(frames, dtype=np.float64)
        y = self.amp * np.sin(idx)
        self.phase = (idx[-1] + inc) % self.twopi
        return y.astype(np.float32)

    def formatted_readouts(self):
        return [f"{self.param_labels[0]}: {self.freq:7.2f} Hz",
                f"{self.param_labels[1]}: {self.amp:.3f}"]

class NoisyLPGenerator(BaseGenerator):
    """
    White noise -> one-pole lowpass.
      p[0] -> cutoff in [100, 8000] Hz (exp)
      p[1] -> level in [0,1]
    """
    param_labels = ["Cutoff (Hz)", "Level"]

    def __init__(self, c_lo=100.0, c_hi=8000.0, seed=0, init_norm_params=None):
        super().__init__(init_norm_params=init_norm_params or [0.5, 0.2])
        self.c_lo = max(1e-3, float(c_lo))
        self.c_hi = max(self.c_lo, float(c_hi))
        self.prev = 0.0
        self.rng = np.random.default_rng(seed)
        self.cut = exp_map01(self.norm_params[0], self.c_lo, self.c_hi)
        self.level = self.norm_params[1]

    def set_params(self, norm_params):
        super().set_params(norm_params)
        self.cut   = exp_map01(self.norm_params[0], self.c_lo, self.c_hi)
        self.level = self.norm_params[1]

    def generate(self, frames, sr):
        if self.level <= 0.0:
            return np.zeros(frames, dtype=np.float32)
        a = 1.0 - np.exp(-2.0 * np.pi * self.cut / sr)
        x = self.rng.standard_normal(frames).astype(np.float64) * 0.5
        y = np.empty_like(x)
        y_prev = float(self.prev)
        for i in range(frames):
            y_prev = y_prev + a * (x[i] - y_prev)
            y[i] = y_prev
        self.prev = y_prev
        return (self.level * y).astype(np.float32)

    def formatted_readouts(self):
        return [f"{self.param_labels[0]}: {self.cut:7.1f} Hz",
                f"{self.param_labels[1]}: {self.level:.3f}"]

# --------- TEMPLATE you can copy for new generators ---------
class MyGeneratorTemplate(BaseGenerator):
    """
    Start here for new DSP ideas. Adjust labels/mapping/state as needed.
      Example mapping:
        p[0] -> something exponential in [lo1, hi1]
        p[1] -> something linear in [lo2, hi2]
      Add/remove params by editing param_labels and init_norm_params.
    """
    param_labels = ["Param A", "Param B"]  # add more names as needed

    def __init__(self, lo1=0.1, hi1=10.0, lo2=0.0, hi2=1.0, init_norm_params=None):
        super().__init__(init_norm_params=init_norm_params or [0.5, 0.5])
        # store ranges
        self.lo1, self.hi1 = float(lo1), float(hi1)
        self.lo2, self.hi2 = float(lo2), float(hi2)
        # persistent DSP state here
        self._state_var = 0.0
        # semantic initialization
        self.val1 = exp_map01(self.norm_params[0], self.lo1, self.hi1)
        self.val2 = self.lo2 + self.norm_params[1]*(self.hi2 - self.lo2)

    def set_params(self, norm_params):
        super().set_params(norm_params)
        self.val1 = exp_map01(self.norm_params[0], self.lo1, self.hi1)
        self.val2 = self.lo2 + self.norm_params[1]*(self.hi2 - self.lo2)

    def generate(self, frames, sr):
        # Replace with your DSP; here we just output silence as a stub.
        return np.zeros(frames, dtype=np.float32)

    def formatted_readouts(self):
        return [f"{self.param_labels[0]}: {self.val1:.3f}",
                f"{self.param_labels[1]}: {self.val2:.3f}"]

In [4]:
# --------- realtime engine (N params) ---------
class RealtimeSynth:
    """
    Realtime audio engine using sounddevice with a pluggable class-based generator.
    Accepts an arbitrary number of normalized params depending on the generator.
    """
    def __init__(self, generator: BaseGenerator, samplerate=48000, blocksize=128, channels=1):
        self.sr = int(samplerate)
        self.blocksize = int(blocksize)
        self.channels = int(channels)
        self._lock = threading.Lock()
        self.gen = generator
        # store normalized params list (initialize from generator)
        self._norm = list(self.gen.norm_params)

        self.stream = sd.OutputStream(
            samplerate=self.sr,
            channels=self.channels,
            dtype='float32',
            blocksize=self.blocksize,
            latency='low',
            callback=self._callback,
        )
        self._running = False

    def set_params(self, norm_list):
        with self._lock:
            # resize/clip to generator length
            n = self.gen.num_params()
            vals = (list(norm_list) + [0.5]*n)[:n]
            self._norm = [float(np.clip(v,0.0,1.0)) for v in vals]

    def set_generator(self, generator: BaseGenerator):
        with self._lock:
            self.gen = generator
            self._norm = list(self.gen.norm_params)

    def start(self):
        if not self._running:
            self.stream.start()
            self._running = True

    def stop(self):
        if self._running:
            self.stream.stop()
            self._running = False

    def close(self):
        try:
            self.stream.close()
        except Exception:
            pass

    def _callback(self, outdata, frames, time, status):
        if status:
            # print("Audio status:", status)
            pass
        with self._lock:
            gen = self.gen
            norm = list(self._norm)
        gen.set_params(norm)
        mono = gen.generate(frames, self.sr)
        if self.channels == 1:
            outdata[:, 0] = mono
        else:
            outdata[:] = np.repeat(mono[:, None], self.channels, axis=1)

# --------- UI that adapts to N params + generator switching ---------
# Register available generators here
GENS = {
    "Sine": lambda: SineGenerator(),
    "Noisy LP": lambda: NoisyLPGenerator(),
    "Template": lambda: MyGeneratorTemplate(),
}

# create initial synth
current_gen = GENS["Sine"]()
synth = RealtimeSynth(generator=current_gen, samplerate=48000, blocksize=128, channels=1)

In [5]:
# dynamic UI builder
sliders_box = VBox()
readouts_box = HBox()
play_toggle = ToggleButton(value=False, description='▶️ Play', layout=Layout(width='120px'))
gen_picker = Dropdown(options=list(GENS.keys()), value="Sine", description="Generator:")

status_line = HTML(value="")

def build_param_ui():
    # build sliders and readouts from current generator
    sliders = []
    readout_labels = []
    n = synth.gen.num_params()
    for i in range(n):
        lab = synth.gen.param_labels[i] if i < len(synth.gen.param_labels) else f"Param {i+1}"
        init = synth.gen.norm_params[i] if i < len(synth.gen.norm_params) else 0.5
        s = FloatSlider(value=float(init), min=0.0, max=1.0, step=0.001,
                        description=lab, layout=Layout(width='420px'))
        def make_on_change(idx):
            def on_change(change):
                if change['name'] == 'value':
                    # update the whole vector
                    values = [w.value for w in sliders_box.children]  # read all current slider values
                    synth.set_params(values)
                    refresh_readouts()
            return on_change
        s.observe(make_on_change(i), names='value')
        sliders.append(s)
        readout_labels.append(Label("", layout=Layout(width='220px')))

    sliders_box.children = sliders
    readouts_box.children = readout_labels
    # push initial param vector
    synth.set_params([w.value for w in sliders])
    refresh_readouts()

def refresh_readouts():
    ro = synth.gen.formatted_readouts()
    # ensure length matches readout labels
    labels = list(readouts_box.children)
    for i, lbl in enumerate(labels):
        lbl.value = ro[i] if i < len(ro) else ""

def on_play(change):
    if change['name'] == 'value':
        if change['new']:
            play_toggle.description = '⏹ Stop'
            synth.start()
            status_line.value = "<i>Audio running…</i>"
        else:
            play_toggle.description = '▶️ Play'
            synth.stop()
            status_line.value = "<i>Audio stopped.</i>"

def on_pick(change):
    if change['name'] == 'value':
        synth.set_generator(GENS[change['new']]())
        build_param_ui()

In [6]:
play_toggle.observe(on_play, names='value')
gen_picker.observe(on_pick, names='value')

# assemble UI
controls = VBox([
    gen_picker,
    sliders_box,
    HBox([play_toggle]),
    HBox([Label("Mapped:"), readouts_box]),
    HTML("<small>Sliders are normalized [0,1]; each generator defines its own mapping & state.<br>"
         "Tip: reduce blocksize for lower latency (risking underruns); increase if you hear clicks.</small>")
])
build_param_ui()
display(controls, status_line)

HTML(value='')